# RQ2b — Effect of Number of Authority Peers
Sweeps n_auth ∈ {1,2,3,4,5} × authority correct/wrong. Shows how authority count and direction interact.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path

RESULTS_DIR = Path("../results_todos/rq2b")

N_AUTH = [1, 2, 3, 4, 5]
CORRECT_CONDS = [f"rq2b_auth{n}_correct" for n in N_AUTH]
WRONG_CONDS   = [f"rq2b_auth{n}_wrong"   for n in N_AUTH]
ALL_CONDS = CORRECT_CONDS + WRONG_CONDS

AUTH_LABELS = {
    **{f"rq2b_auth{n}_correct": f"{n} auth (correct)" for n in N_AUTH},
    **{f"rq2b_auth{n}_wrong":   f"{n} auth (wrong)"   for n in N_AUTH},
}

per_seed_dfs = []
for csv_path in sorted(RESULTS_DIR.rglob("*.metrics.csv")):
    if "_all_seeds" in csv_path.name:
        continue
    parts = csv_path.parts
    if not any(p.startswith("seed_") for p in parts):
        continue
    df = pd.read_csv(csv_path)
    df = df[df["condition"].isin(ALL_CONDS + ["ALL"])]
    if not df.empty:
        per_seed_dfs.append(df)

all_per_seed = pd.concat(per_seed_dfs, ignore_index=True) if per_seed_dfs else pd.DataFrame()
combined = all_per_seed.groupby(["model", "condition"]).mean(numeric_only=True).reset_index()

MODELS = sorted(combined["model"].unique())
PALETTE = sns.color_palette("tab10", len(MODELS))
MODEL_COLORS = dict(zip(MODELS, PALETTE))

print(f"Models: {MODELS}")
print(f"Conditions: {sorted(combined['condition'].unique())}")


## Per-model metrics tables

In [ ]:
display_cols = ["condition", "n", "acc_r1", "acc_r2", "delta_acc",
                "rev_pct", "harm_pct", "ben_pct", "mean_dC"]

for model in MODELS:
    df_m = combined[(combined["model"] == model) & (combined["condition"].isin(ALL_CONDS))].copy()
    present = [c for c in ALL_CONDS if c in df_m["condition"].values]
    df_m["condition"] = pd.Categorical(df_m["condition"], categories=present, ordered=True)
    df_m = df_m.sort_values("condition")[display_cols].reset_index(drop=True)
    print(f"\n{'─'*60}\n  {model.upper()}\n{'─'*60}")
    display(df_m.style
        .format({"n": lambda x: f"{int(x)}" if pd.notna(x) and x == int(x) else (f"{x:.2f}" if pd.notna(x) else "N/A"),
                 **{c: "{:.1f}" for c in ["acc_r1","acc_r2","delta_acc","rev_pct","harm_pct","ben_pct","mean_dC"]}}, na_rep="N/A")
        .background_gradient(subset=["delta_acc"], cmap="RdYlGn", vmin=-30, vmax=30)
        .background_gradient(subset=["harm_pct"], cmap="Reds", vmin=0, vmax=100)
        .background_gradient(subset=["ben_pct"], cmap="Greens", vmin=0, vmax=100)
        .set_caption(model))


## Δ Accuracy vs n_auth — correct vs wrong authority (line plot)

In [ ]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(5*len(MODELS), 5), sharey=True)
if len(MODELS) == 1:
    axes = [axes]

for ax, model in zip(axes, MODELS):
    df_m = combined[combined["model"] == model]
    for side, conds, color, label in [
        ("correct", CORRECT_CONDS, "#2ca02c", "Authority correct"),
        ("wrong",   WRONG_CONDS,   "#d62728", "Authority wrong"),
    ]:
        df_s = df_m[df_m["condition"].isin(conds)].copy()
        df_s["n_auth"] = df_s["condition"].str.extract(r"auth(\d+)").astype(int)
        df_s = df_s.sort_values("n_auth")
        ax.plot(df_s["n_auth"], df_s["delta_acc"], marker="o", color=color, label=label)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(model, fontsize=12, fontweight="bold")
    ax.set_xlabel("n_auth")
    ax.set_ylabel("\u0394 Accuracy (pp)" if ax == axes[0] else "")
    ax.set_xticks(N_AUTH)
    ax.legend(fontsize=8)
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v:+.0f}pp"))

plt.suptitle("\u0394 Accuracy vs Number of Authority Peers", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "rq2b_delta_acc_vs_nauth.png", dpi=150, bbox_inches="tight")
plt.show()


## Δ Accuracy heatmap — models × conditions

In [ ]:
df_plot = combined[combined["condition"].isin(ALL_CONDS)]
pivot = df_plot.pivot_table(index="condition", columns="model", values="delta_acc")
pivot = pivot.reindex([c for c in ALL_CONDS if c in pivot.index])

fig, ax = plt.subplots(figsize=(max(6, 2.5*len(MODELS)), 7))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="RdYlGn", center=0, vmin=-55, vmax=55,
            linewidths=0.5, ax=ax, annot_kws={"size": 9})
ax.set_title("\u0394 Accuracy by Model and Authority Condition", fontsize=13, fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("")
# FIX: human-readable y-tick labels
ax.set_yticklabels([AUTH_LABELS.get(c, c) for c in pivot.index], rotation=0)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "rq2b_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
